# DriftGuard — Structural Configuration Diff Generation

This notebook converts file-level Git changes into field-level configuration changes.

It:

1. Parses YAML, JSON, Terraform/HCL, nginx, INI, TOML, Dockerfile and generic configuration files.
2. Flattens nested configuration structures into stable field paths.
3. Compares configuration values before and after each commit.
4. Produces added, deleted and modified field-level Diff Objects.
5. Uses a line-level fallback when structural parsing fails.
6. Preserves repository-disjoint train, validation and test splits.
7. Generates parser-quality and data-integrity reports.
   

In [1]:
import os
import re
import sys
import json
import gzip
import hashlib
import difflib
import configparser
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import pandas as pd
import yaml
from tqdm.auto import tqdm

try:
    import tomllib
except ImportError:
    import tomli as tomllib


print("=" * 72)
print("DRIFTGUARD — STRUCTURAL DIFF GENERATION")
print("=" * 72)

current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

CONFIGS_DIR = PROJECT_ROOT / "configs"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
LOGS_DIR = PROJECT_ROOT / "logs"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Python version  :", sys.version.split()[0])
print("Python path     :", sys.executable)
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "Not detected"))
print("Project root    :", PROJECT_ROOT)
print("Raw data        :", RAW_DATA_DIR)
print("Interim data    :", INTERIM_DATA_DIR)

DRIFTGUARD — STRUCTURAL DIFF GENERATION
Python version  : 3.11.15
Python path     : C:\Users\Lenovo\anaconda3\envs\driftguard\python.exe
Conda environment: driftguard
Project root    : C:\Users\Lenovo\Desktop\DriftGuard
Raw data        : C:\Users\Lenovo\Desktop\DriftGuard\data\raw
Interim data    : C:\Users\Lenovo\Desktop\DriftGuard\data\interim


In [2]:
project_settings_path = CONFIGS_DIR / "project_settings.json"
repository_config_path = CONFIGS_DIR / "repositories.json"

if not project_settings_path.exists():
    raise FileNotFoundError(
        f"Missing project settings:\n{project_settings_path}"
    )

if not repository_config_path.exists():
    raise FileNotFoundError(
        f"Missing repository settings:\n{repository_config_path}"
    )

with project_settings_path.open("r", encoding="utf-8") as file:
    PROJECT_SETTINGS = json.load(file)

with repository_config_path.open("r", encoding="utf-8") as file:
    repository_configuration = json.load(file)

REPOSITORIES = repository_configuration["repositories"]

CONFIG_EXTENSIONS = {
    extension.lower()
    for extension in PROJECT_SETTINGS["config_extensions"]
}

print("Repositories loaded:", len(REPOSITORIES))

display(
    pd.DataFrame(REPOSITORIES)[
        ["name", "split", "primary_config_types"]
    ]
)

Repositories loaded: 6


,name,split,primary_config_types
0,microservices_demo,train,"[kubernetes, helm, kustomize, terraform]"
1,kube_prometheus,train,"[kubernetes, jsonnet, yaml]"
2,terraform_aws_vpc,train,[terraform]
3,kubernetes_examples,validation,"[kubernetes, yaml]"
4,ansible_examples,test,"[ansible, nginx, yaml]"
5,terraform_eks_blueprints,test,"[terraform, kubernetes]"


In [3]:
extraction_file_records = []

for repository in REPOSITORIES:
    repository_name = repository["name"]

    extraction_path = (
        RAW_DATA_DIR
        / f"{repository_name}_configuration_changes.jsonl.gz"
    )

    extraction_file_records.append(
        {
            "repository": repository_name,
            "split": repository["split"],
            "file_exists": extraction_path.exists(),
            "file_size_mb": (
                round(
                    extraction_path.stat().st_size
                    / (1024 ** 2),
                    2,
                )
                if extraction_path.exists()
                else None
            ),
            "path": str(extraction_path),
        }
    )

extraction_files_df = pd.DataFrame(extraction_file_records)

display(extraction_files_df)

if not extraction_files_df["file_exists"].all():
    missing_repositories = extraction_files_df.loc[
        ~extraction_files_df["file_exists"],
        "repository",
    ].tolist()

    raise FileNotFoundError(
        "Missing Notebook 02 outputs for: "
        + ", ".join(missing_repositories)
    )

print("\nAll Notebook 02 extraction files are available.")

,repository,split,file_exists,file_size_mb,path
0,microservices_demo,train,True,38.84,C:\Users\Lenovo\Desktop\DriftGuard\data\raw\mi...
1,kube_prometheus,train,True,79.13,C:\Users\Lenovo\Desktop\DriftGuard\data\raw\ku...
2,terraform_aws_vpc,train,True,2.32,C:\Users\Lenovo\Desktop\DriftGuard\data\raw\te...
3,kubernetes_examples,validation,True,1.09,C:\Users\Lenovo\Desktop\DriftGuard\data\raw\ku...
4,ansible_examples,test,True,0.36,C:\Users\Lenovo\Desktop\DriftGuard\data\raw\an...
5,terraform_eks_blueprints,test,True,6.47,C:\Users\Lenovo\Desktop\DriftGuard\data\raw\te...



All Notebook 02 extraction files are available.


In [4]:
STRUCTURAL_DIFF_SETTINGS = {
    "force_regenerate": False,

    # Prevent abnormally large parsed structures from exhausting memory.
    "max_fields_per_snapshot": 5_000,

    # Prevent one generated or vendor file from dominating the dataset.
    "max_diffs_per_file_change": 750,

    # Long values are truncated in model-ready records.
    "max_value_characters": 4_000,

    # Maximum lines processed by the fallback SequenceMatcher.
    "max_fallback_lines": 5_000,

    # Comment-only changes do not produce semantic diffs.
    "ignore_comment_only_changes": True,
}

settings_output_path = (
    CONFIGS_DIR / "structural_diff_settings.json"
)

with settings_output_path.open("w", encoding="utf-8") as file:
    json.dump(
        STRUCTURAL_DIFF_SETTINGS,
        file,
        indent=2,
    )

print(json.dumps(STRUCTURAL_DIFF_SETTINGS, indent=2))
print("\nSaved to:", settings_output_path)

{
  "force_regenerate": false,
  "max_fields_per_snapshot": 5000,
  "max_diffs_per_file_change": 750,
  "max_value_characters": 4000,
  "max_fallback_lines": 5000,
  "ignore_comment_only_changes": true
}

Saved to: C:\Users\Lenovo\Desktop\DriftGuard\configs\structural_diff_settings.json


In [5]:
class FieldLimitExceeded(Exception):
    """Raised when a parsed snapshot contains too many fields."""


def stable_sha256(*values):
    """Create a stable SHA-256 hash from several values."""

    combined = "|".join(
        "" if value is None else str(value)
        for value in values
    )

    return hashlib.sha256(
        combined.encode("utf-8", errors="replace")
    ).hexdigest()


def convert_to_plain_data(value):
    """Convert parser-specific values into JSON-safe Python values."""

    if isinstance(value, dict):
        return {
            str(key): convert_to_plain_data(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            convert_to_plain_data(item)
            for item in value
        ]

    if isinstance(value, set):
        return sorted(
            convert_to_plain_data(item)
            for item in value
        )

    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    if hasattr(value, "isoformat") and not isinstance(value, str):
        try:
            return value.isoformat()
        except Exception:
            return str(value)

    if value is None or isinstance(
        value,
        (str, int, float, bool),
    ):
        return value

    return str(value)


def canonical_value(value):
    """Convert a value into a stable comparison representation."""

    plain_value = convert_to_plain_data(value)

    if isinstance(plain_value, str):
        return plain_value

    return json.dumps(
        plain_value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def value_type_name(value):
    """Return a consistent type label."""

    if value is None:
        return "null"

    if isinstance(value, bool):
        return "boolean"

    if isinstance(value, int):
        return "integer"

    if isinstance(value, float):
        return "float"

    if isinstance(value, list):
        return "list"

    if isinstance(value, dict):
        return "object"

    return "string"


def truncate_value(value, maximum_characters=None):
    """Serialize and truncate a value for storage."""

    if maximum_characters is None:
        maximum_characters = STRUCTURAL_DIFF_SETTINGS[
            "max_value_characters"
        ]

    if value is None:
        return None

    serialized = canonical_value(value)

    if len(serialized) <= maximum_characters:
        return serialized

    return (
        serialized[:maximum_characters]
        + " ... [TRUNCATED]"
    )


def normalize_path_component(value):
    """Normalize a key for use inside a field path."""

    text = str(value).strip()

    if not text:
        return "<empty>"

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    return text

In [7]:
def strip_inline_comment(line):
    """
    Remove common inline comments while attempting to preserve quoted text
    and URLs such as https://example.com.
    """

    if line is None:
        return ""

    in_single_quote = False
    in_double_quote = False
    escaped = False

    index = 0

    while index < len(line):
        character = line[index]

        if escaped:
            escaped = False
            index += 1
            continue

        if character == "\\":
            escaped = True
            index += 1
            continue

        if character == "'" and not in_double_quote:
            in_single_quote = not in_single_quote

        elif character == '"' and not in_single_quote:
            in_double_quote = not in_double_quote

        elif not in_single_quote and not in_double_quote:
            if character == "#":
                return line[:index].rstrip()

            if (
                character == "/"
                and index + 1 < len(line)
                and line[index + 1] == "/"
            ):
                previous_character = (
                    line[index - 1]
                    if index > 0
                    else ""
                )

                if previous_character != ":":
                    return line[:index].rstrip()

        index += 1

    return line.rstrip()


def meaningful_lines(text, maximum_lines=None):
    """Return non-empty, non-comment lines with original line numbers."""

    if maximum_lines is None:
        maximum_lines = STRUCTURAL_DIFF_SETTINGS[
            "max_fallback_lines"
        ]

    if text is None:
        return []

    records = []

    for line_number, raw_line in enumerate(
        text.splitlines(),
        start=1,
    ):
        cleaned_line = strip_inline_comment(raw_line).strip()

        if not cleaned_line:
            continue

        records.append(
            {
                "line_number": line_number,
                "content": cleaned_line,
            }
        )

        if len(records) >= maximum_lines:
            break

    return records

In [8]:
def flatten_structure(
    value,
    prefix="$",
    output=None,
    maximum_fields=None,
):
    """
    Flatten dictionaries and lists into field paths.

    Example:
    spec:
      replicas: 3

    becomes:
    $.spec.replicas -> 3
    """

    if output is None:
        output = {}

    if maximum_fields is None:
        maximum_fields = STRUCTURAL_DIFF_SETTINGS[
            "max_fields_per_snapshot"
        ]

    if len(output) >= maximum_fields:
        raise FieldLimitExceeded(
            f"Parsed structure exceeded {maximum_fields:,} fields."
        )

    value = convert_to_plain_data(value)

    if isinstance(value, dict):
        if not value:
            output[prefix] = {}

        for key in sorted(
            value.keys(),
            key=lambda item: str(item),
        ):
            child_prefix = (
                f"{prefix}.{normalize_path_component(key)}"
            )

            flatten_structure(
                value=value[key],
                prefix=child_prefix,
                output=output,
                maximum_fields=maximum_fields,
            )

    elif isinstance(value, list):
        if not value:
            output[prefix] = []

        for index, item in enumerate(value):
            child_prefix = f"{prefix}[{index}]"

            flatten_structure(
                value=item,
                prefix=child_prefix,
                output=output,
                maximum_fields=maximum_fields,
            )

    else:
        output[prefix] = value

    return output

In [10]:
def parse_yaml_configuration(text):
    """Parse one or more YAML documents."""

    documents = list(
        yaml.safe_load_all(text)
    )

    if len(documents) == 0:
        return {}

    if len(documents) == 1:
        return documents[0]

    return {
        "documents": documents,
    }


def parse_json_configuration(text):
    """Parse JSON configuration."""

    return json.loads(text)


def parse_toml_configuration(text):
    """Parse TOML configuration."""

    return tomllib.loads(text)


def parse_ini_configuration(text):
    """Parse INI/CFG configuration."""

    parser = configparser.ConfigParser(
        interpolation=None,
        strict=False,
        allow_no_value=True,
    )

    parser.optionxform = str
    parser.read_string(text)

    parsed = {}

    if parser.defaults():
        parsed["DEFAULT"] = dict(parser.defaults())

    for section in parser.sections():
        section_values = {}

        for key, value in parser.items(
            section,
            raw=True,
        ):
            section_values[key] = value

        parsed[section] = section_values

    if not parsed and text.strip():
        raise ValueError(
            "No INI sections or assignments were detected."
        )

    return parsed

In [11]:
def docker_logical_lines(text):
    """Join Dockerfile continuation lines."""

    logical_lines = []
    buffer = ""

    for raw_line in text.splitlines():
        stripped_line = raw_line.strip()

        if not stripped_line:
            continue

        if stripped_line.startswith("#"):
            continue

        if stripped_line.endswith("\\"):
            buffer += stripped_line[:-1].rstrip() + " "
            continue

        buffer += stripped_line
        logical_lines.append(buffer.strip())
        buffer = ""

    if buffer.strip():
        logical_lines.append(buffer.strip())

    return logical_lines


def parse_dockerfile_configuration(text):
    """Parse Dockerfile instructions into indexed fields."""

    parsed = {}
    instruction_counts = Counter()

    for logical_line in docker_logical_lines(text):
        parts = logical_line.split(
            None,
            maxsplit=1,
        )

        instruction = parts[0].upper()
        argument = parts[1] if len(parts) > 1 else ""

        index = instruction_counts[instruction]
        instruction_counts[instruction] += 1

        parsed[
            f"instructions.{instruction}[{index}]"
        ] = argument

    if not parsed and text.strip():
        raise ValueError(
            "No Dockerfile instructions were detected."
        )

    return parsed

In [12]:
def sanitize_block_header(header):
    """Convert a block declaration into a path-safe label."""

    header = re.sub(r"\s+", " ", header.strip())
    header = header.replace('"', "")
    header = header.replace("'", "")

    return header or "<anonymous_block>"


def parse_nginx_configuration(text):
    """
    Parse common nginx-style blocks and directives.

    This is intentionally lightweight. Unsupported syntax is handled by
    the line-level fallback instead of causing the pipeline to fail.
    """

    parsed = {}
    block_stack = []
    block_counts = Counter()
    directive_counts = Counter()

    for raw_line in text.splitlines():
        line = strip_inline_comment(raw_line).strip()

        if not line:
            continue

        while line.startswith("}"):
            if block_stack:
                block_stack.pop()

            line = line[1:].strip()

        if not line:
            continue

        if line.endswith("{"):
            block_header = sanitize_block_header(
                line[:-1]
            )

            parent_path = ".".join(block_stack)

            block_key = (
                parent_path,
                block_header,
            )

            block_index = block_counts[block_key]
            block_counts[block_key] += 1

            block_label = (
                f"{block_header}[{block_index}]"
            )

            block_stack.append(block_label)
            continue

        line = line.rstrip(";").strip()

        if not line:
            continue

        parts = line.split(None, maxsplit=1)

        directive_name = parts[0]
        directive_value = (
            parts[1] if len(parts) > 1 else ""
        )

        base_path = ".".join(
            block_stack + [directive_name]
        )

        directive_index = directive_counts[base_path]
        directive_counts[base_path] += 1

        parsed[
            f"{base_path}[{directive_index}]"
        ] = directive_value

    if not parsed and text.strip():
        raise ValueError(
            "No nginx directives were detected."
        )

    return parsed

In [13]:
def remove_hcl_block_comments(text):
    """Remove /* ... */ comments from HCL text."""

    return re.sub(
        r"/\*.*?\*/",
        "",
        text,
        flags=re.DOTALL,
    )


def bracket_balance(text):
    """Calculate approximate collection-bracket balance."""

    balance = 0
    in_single_quote = False
    in_double_quote = False
    escaped = False

    for character in text:
        if escaped:
            escaped = False
            continue

        if character == "\\":
            escaped = True
            continue

        if character == "'" and not in_double_quote:
            in_single_quote = not in_single_quote
            continue

        if character == '"' and not in_single_quote:
            in_double_quote = not in_double_quote
            continue

        if in_single_quote or in_double_quote:
            continue

        if character in "[(":
            balance += 1
        elif character in "])":
            balance -= 1

    return balance


def parse_hcl_configuration(text):
    """
    Parse common Terraform/HCL blocks and assignments.

    Complex expressions remain as canonical strings.
    """

    text = remove_hcl_block_comments(text)

    parsed = {}
    block_stack = []
    block_counts = Counter()
    assignment_counts = Counter()

    lines = text.splitlines()
    line_index = 0

    while line_index < len(lines):
        line = strip_inline_comment(
            lines[line_index]
        ).strip()

        line_index += 1

        if not line:
            continue

        while line.startswith("}"):
            if block_stack:
                block_stack.pop()

            line = line[1:].strip()

        if not line:
            continue

        if line.endswith("{") and "=" not in line:
            header = sanitize_block_header(
                line[:-1]
            )

            parent_path = ".".join(block_stack)
            block_key = (
                parent_path,
                header,
            )

            block_number = block_counts[block_key]
            block_counts[block_key] += 1

            block_stack.append(
                f"{header}[{block_number}]"
            )

            continue

        assignment_match = re.match(
            r"^([A-Za-z0-9_.-]+)\s*=\s*(.*)$",
            line,
        )

        if not assignment_match:
            continue

        assignment_name = assignment_match.group(1)
        assignment_value = assignment_match.group(2).strip()

        while (
            bracket_balance(assignment_value) > 0
            and line_index < len(lines)
        ):
            continuation = strip_inline_comment(
                lines[line_index]
            ).strip()

            line_index += 1

            if continuation:
                assignment_value += " " + continuation

        assignment_value = re.sub(
            r"\s+",
            " ",
            assignment_value,
        ).strip()

        base_path = ".".join(
            block_stack + [assignment_name]
        )

        assignment_number = assignment_counts[
            base_path
        ]

        assignment_counts[base_path] += 1

        parsed[
            f"{base_path}[{assignment_number}]"
        ] = assignment_value

    if not parsed and text.strip():
        raise ValueError(
            "No HCL blocks or assignments were detected."
        )

    return parsed

In [14]:
def parse_generic_key_value_configuration(text):
    """
    Parse common key=value, key:value and directive-style files.
    """

    parsed = {}
    current_section = "root"
    key_counts = Counter()

    for raw_line in text.splitlines():
        line = strip_inline_comment(raw_line).strip()

        if not line:
            continue

        if line.startswith("[") and line.endswith("]"):
            current_section = line[1:-1].strip()
            continue

        if line in {"{", "}"}:
            continue

        key = None
        value = None

        if "=" in line:
            key, value = line.split(
                "=",
                maxsplit=1,
            )

        elif ":" in line and not re.match(
            r"^[A-Za-z]+://",
            line,
        ):
            key, value = line.split(
                ":",
                maxsplit=1,
            )

        else:
            parts = line.rstrip(";").split(
                None,
                maxsplit=1,
            )

            if len(parts) == 2:
                key, value = parts

        if key is None:
            continue

        key = key.strip()
        value = value.strip().rstrip(";")

        if not key:
            continue

        base_path = f"{current_section}.{key}"
        key_index = key_counts[base_path]
        key_counts[base_path] += 1

        parsed[
            f"{base_path}[{key_index}]"
        ] = value

    if not parsed and text.strip():
        raise ValueError(
            "No key-value configuration entries were detected."
        )

    return parsed

In [15]:
def select_parser_name(file_path, configuration_type):
    """Select the best parser using file path and inferred type."""

    normalized_path = (
        str(file_path or "")
        .replace("\\", "/")
        .lower()
    )

    filename = Path(normalized_path).name
    suffix = Path(normalized_path).suffix.lower()

    if filename in {
        "dockerfile",
        "containerfile",
    }:
        return "dockerfile"

    if configuration_type == "docker":
        return "dockerfile"

    if configuration_type in {
        "kubernetes",
        "helm",
        "kustomize",
        "ansible",
        "yaml",
        "docker_compose",
    }:
        return "yaml"

    if suffix in {".yaml", ".yml"}:
        return "yaml"

    if configuration_type == "terraform":
        return "hcl"

    if suffix in {".tf", ".tfvars", ".hcl"}:
        return "hcl"

    if configuration_type == "json" or suffix == ".json":
        return "json"

    if configuration_type == "toml" or suffix == ".toml":
        return "toml"

    if (
        configuration_type == "nginx"
        or "nginx" in normalized_path
        or filename == "nginx.conf"
    ):
        return "nginx"

    if suffix in {".ini", ".cfg"}:
        return "ini"

    return "generic_key_value"


PARSER_FUNCTIONS = {
    "yaml": parse_yaml_configuration,
    "json": parse_json_configuration,
    "toml": parse_toml_configuration,
    "ini": parse_ini_configuration,
    "dockerfile": parse_dockerfile_configuration,
    "nginx": parse_nginx_configuration,
    "hcl": parse_hcl_configuration,
    "generic_key_value":
        parse_generic_key_value_configuration,
}


def parse_configuration_snapshot(
    text,
    file_path,
    configuration_type,
):
    """Parse and flatten one configuration snapshot."""

    parser_name = select_parser_name(
        file_path=file_path,
        configuration_type=configuration_type,
    )

    if text is None or text.strip() == "":
        return {
            "success": True,
            "parser_name": parser_name,
            "flat_values": {},
            "field_count": 0,
            "error": None,
        }

    parser_function = PARSER_FUNCTIONS[parser_name]

    try:
        parsed_structure = parser_function(text)

        flattened = flatten_structure(
            parsed_structure,
            maximum_fields=STRUCTURAL_DIFF_SETTINGS[
                "max_fields_per_snapshot"
            ],
        )

        return {
            "success": True,
            "parser_name": parser_name,
            "flat_values": flattened,
            "field_count": len(flattened),
            "error": None,
        }

    except Exception as error:
        return {
            "success": False,
            "parser_name": parser_name,
            "flat_values": {},
            "field_count": 0,
            "error": (
                f"{type(error).__name__}: {error}"
            ),
        }

In [16]:
parser_test_cases = [
    {
        "name": "yaml",
        "path": "deployment.yaml",
        "configuration_type": "kubernetes",
        "text": """
apiVersion: apps/v1
kind: Deployment
spec:
  replicas: 3
  template:
    spec:
      containers:
        - name: frontend
          image: nginx:1.25
""",
    },
    {
        "name": "json",
        "path": "config.json",
        "configuration_type": "json",
        "text": """
{
  "authentication": {
    "enabled": true
  },
  "port": 443
}
""",
    },
    {
        "name": "terraform",
        "path": "main.tf",
        "configuration_type": "terraform",
        "text": """
resource "aws_security_group" "web" {
  name = "web-security-group"

  ingress {
    from_port = 443
    to_port   = 443
    cidr_blocks = ["10.0.0.0/8"]
  }
}
""",
    },
    {
        "name": "nginx",
        "path": "nginx.conf",
        "configuration_type": "nginx",
        "text": """
server {
    listen 443 ssl;
    server_name example.com;

    location / {
        proxy_pass http://backend;
    }
}
""",
    },
    {
        "name": "ini",
        "path": "application.ini",
        "configuration_type": "ini",
        "text": """
[server]
port = 443
authentication = enabled
""",
    },
    {
        "name": "dockerfile",
        "path": "Dockerfile",
        "configuration_type": "docker",
        "text": """
FROM python:3.11
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt
CMD ["python", "app.py"]
""",
    },
]

parser_test_results = []

for test_case in parser_test_cases:
    result = parse_configuration_snapshot(
        text=test_case["text"],
        file_path=test_case["path"],
        configuration_type=test_case[
            "configuration_type"
        ],
    )

    parser_test_results.append(
        {
            "test": test_case["name"],
            "success": result["success"],
            "parser": result["parser_name"],
            "field_count": result["field_count"],
            "error": result["error"],
        }
    )

parser_test_df = pd.DataFrame(parser_test_results)

display(parser_test_df)

assert parser_test_df["success"].all()
assert (parser_test_df["field_count"] > 0).all()

print("All parser tests: PASSED")

,test,success,parser,field_count,error
0,yaml,True,yaml,5,None
1,json,True,json,2,None
2,terraform,True,hcl,4,None
3,nginx,True,nginx,3,None
4,ini,True,ini,2,None
5,dockerfile,True,dockerfile,5,None


All parser tests: PASSED


In [17]:
def compare_flat_structures(
    old_values,
    new_values,
):
    """Compare flattened structures and create field-level changes."""

    changes = []

    all_field_paths = sorted(
        set(old_values.keys())
        | set(new_values.keys())
    )

    for field_path in all_field_paths:
        old_exists = field_path in old_values
        new_exists = field_path in new_values

        old_value = (
            old_values.get(field_path)
            if old_exists
            else None
        )

        new_value = (
            new_values.get(field_path)
            if new_exists
            else None
        )

        if not old_exists and new_exists:
            operation = "added"

        elif old_exists and not new_exists:
            operation = "deleted"

        elif canonical_value(old_value) != canonical_value(
            new_value
        ):
            operation = "modified"

        else:
            continue

        changes.append(
            {
                "operation": operation,
                "field_path": field_path,
                "old_value": old_value,
                "new_value": new_value,
                "old_value_type": (
                    value_type_name(old_value)
                    if old_exists
                    else "missing"
                ),
                "new_value_type": (
                    value_type_name(new_value)
                    if new_exists
                    else "missing"
                ),
            }
        )

    return changes

In [18]:
def create_line_fallback_diffs(
    source_before,
    source_after,
):
    """Generate aligned line-level changes when parsing fails."""

    old_line_records = meaningful_lines(source_before)
    new_line_records = meaningful_lines(source_after)

    old_contents = [
        record["content"]
        for record in old_line_records
    ]

    new_contents = [
        record["content"]
        for record in new_line_records
    ]

    sequence_matcher = difflib.SequenceMatcher(
        a=old_contents,
        b=new_contents,
        autojunk=False,
    )

    changes = []

    for (
        opcode,
        old_start,
        old_end,
        new_start,
        new_end,
    ) in sequence_matcher.get_opcodes():

        if opcode == "equal":
            continue

        if opcode == "replace":
            old_block = old_line_records[
                old_start:old_end
            ]

            new_block = new_line_records[
                new_start:new_end
            ]

            paired_count = min(
                len(old_block),
                len(new_block),
            )

            for pair_index in range(paired_count):
                old_record = old_block[pair_index]
                new_record = new_block[pair_index]

                changes.append(
                    {
                        "operation": "modified",
                        "field_path": (
                            "__line__."
                            f"old[{old_record['line_number']}]"
                            f".new[{new_record['line_number']}]"
                        ),
                        "old_value":
                            old_record["content"],
                        "new_value":
                            new_record["content"],
                        "old_value_type": "string",
                        "new_value_type": "string",
                    }
                )

            for old_record in old_block[paired_count:]:
                changes.append(
                    {
                        "operation": "deleted",
                        "field_path": (
                            "__line__."
                            f"old[{old_record['line_number']}]"
                        ),
                        "old_value":
                            old_record["content"],
                        "new_value": None,
                        "old_value_type": "string",
                        "new_value_type": "missing",
                    }
                )

            for new_record in new_block[paired_count:]:
                changes.append(
                    {
                        "operation": "added",
                        "field_path": (
                            "__line__."
                            f"new[{new_record['line_number']}]"
                        ),
                        "old_value": None,
                        "new_value":
                            new_record["content"],
                        "old_value_type": "missing",
                        "new_value_type": "string",
                    }
                )

        elif opcode == "delete":
            for old_record in old_line_records[
                old_start:old_end
            ]:
                changes.append(
                    {
                        "operation": "deleted",
                        "field_path": (
                            "__line__."
                            f"old[{old_record['line_number']}]"
                        ),
                        "old_value":
                            old_record["content"],
                        "new_value": None,
                        "old_value_type": "string",
                        "new_value_type": "missing",
                    }
                )

        elif opcode == "insert":
            for new_record in new_line_records[
                new_start:new_end
            ]:
                changes.append(
                    {
                        "operation": "added",
                        "field_path": (
                            "__line__."
                            f"new[{new_record['line_number']}]"
                        ),
                        "old_value": None,
                        "new_value":
                            new_record["content"],
                        "old_value_type": "missing",
                        "new_value_type": "string",
                    }
                )

    return changes

In [22]:
old_example = """
apiVersion: apps/v1
kind: Deployment
spec:
  replicas: 2
  template:
    spec:
      containers:
        - name: frontend
          image: nginx:1.24
"""

new_example = """
apiVersion: apps/v1
kind: Deployment
spec:
  replicas: 4
  template:
    spec:
      containers:
        - name: frontend
          image: nginx:1.25
          securityContext:
            privileged: true
"""

old_parsed = parse_configuration_snapshot(
    text=old_example,
    file_path="deployment.yaml",
    configuration_type="kubernetes",
)

new_parsed = parse_configuration_snapshot(
    text=new_example,
    file_path="deployment.yaml",
    configuration_type="kubernetes",
)

example_diffs = compare_flat_structures(
    old_values=old_parsed["flat_values"],
    new_values=new_parsed["flat_values"],
)

example_diff_df = pd.DataFrame(example_diffs)

display(example_diff_df)

assert len(example_diffs) == 3
assert set(example_diff_df["operation"]).issubset(
    {"added", "deleted", "modified"}
)

print("Structural comparison test: PASSED")

,operation,field_path,old_value,new_value,old_value_type,new_value_type
0,modified,$.spec.replicas,2,4,integer,integer
1,modified,$.spec.template.spec.containers[0].image,nginx:1.24,nginx:1.25,string,string
2,added,$.spec.template.spec.containers[0].securityCon...,None,True,missing,boolean


Structural comparison test: PASSED


In [23]:
print("=" * 72)
print("Loading structural-diff extraction function...")
print("=" * 72)


def generate_structural_diffs_for_repository(
    repository_config,
    force_regenerate=False,
):
    """Generate field-level diffs for one repository with live progress."""

    repository_name = repository_config["name"]
    repository_split = repository_config["split"]

    input_path = (
        RAW_DATA_DIR
        / f"{repository_name}_configuration_changes.jsonl.gz"
    )

    extraction_summary_path = (
        RAW_DATA_DIR
        / f"{repository_name}_extraction_summary.json"
    )

    output_path = (
        INTERIM_DATA_DIR
        / f"{repository_name}_structural_diffs.jsonl.gz"
    )

    summary_path = (
        INTERIM_DATA_DIR
        / f"{repository_name}_structural_diff_summary.json"
    )

    temporary_output_path = Path(
        str(output_path) + ".tmp"
    )

    print(f"\nRepository : {repository_name}")
    print(f"Split      : {repository_split}")
    print(f"Input      : {input_path.name}")
    print(f"Output     : {output_path.name}")

    if (
        output_path.exists()
        and summary_path.exists()
        and not force_regenerate
    ):
        print(
            f"Status     : Existing output found. "
            f"Skipping {repository_name}."
        )

        with summary_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Missing extraction file: {input_path}"
        )

    # Read the expected number of file changes from Notebook 02 summary.
    expected_file_changes = None

    if extraction_summary_path.exists():
        try:
            with extraction_summary_path.open(
                "r",
                encoding="utf-8",
            ) as file:
                extraction_summary = json.load(file)

            expected_file_changes = extraction_summary.get(
                "extracted_configuration_change_records"
            )

        except Exception as error:
            print(
                "Could not read extraction summary:",
                type(error).__name__,
                error,
            )

    print(
        "Expected file changes:",
        (
            f"{expected_file_changes:,}"
            if expected_file_changes is not None
            else "Unknown"
        ),
    )

    file_changes_processed = 0
    file_changes_with_diffs = 0
    file_changes_without_semantic_diffs = 0
    generated_diff_records = 0
    capped_file_changes = 0
    processing_errors = 0

    parser_counts = Counter()
    parser_mode_counts = Counter()
    operation_counts = Counter()
    configuration_type_counts = Counter()
    skip_reasons = Counter()
    parser_error_types = Counter()

    if temporary_output_path.exists():
        temporary_output_path.unlink()

    print("Status     : Processing started")

    with gzip.open(
        input_path,
        mode="rt",
        encoding="utf-8",
    ) as input_file, gzip.open(
        temporary_output_path,
        mode="wt",
        encoding="utf-8",
    ) as output_file:

        progress_bar = tqdm(
            input_file,
            total=expected_file_changes,
            desc=f"{repository_name}",
            unit="file",
            dynamic_ncols=True,
        )

        for line in progress_bar:
            file_changes_processed += 1

            try:
                source_record = json.loads(line)

                source_before = source_record.get(
                    "source_before"
                )

                source_after = source_record.get(
                    "source_after"
                )

                if (
                    source_record.get(
                        "source_before_sha256"
                    )
                    == source_record.get(
                        "source_after_sha256"
                    )
                    and source_before is not None
                    and source_after is not None
                ):
                    skip_reasons[
                        "identical_content_hash"
                    ] += 1

                    progress_bar.set_postfix(
                        diffs=generated_diff_records,
                        structured=parser_mode_counts.get(
                            "structured",
                            0,
                        ),
                        fallback=parser_mode_counts.get(
                            "line_fallback",
                            0,
                        ),
                        errors=processing_errors,
                    )

                    continue

                effective_path = source_record[
                    "effective_path"
                ]

                configuration_type = source_record[
                    "configuration_type"
                ]

                old_parse = parse_configuration_snapshot(
                    text=source_before,
                    file_path=effective_path,
                    configuration_type=configuration_type,
                )

                new_parse = parse_configuration_snapshot(
                    text=source_after,
                    file_path=effective_path,
                    configuration_type=configuration_type,
                )

                parser_name = old_parse["parser_name"]
                parser_counts[parser_name] += 1

                if (
                    old_parse["success"]
                    and new_parse["success"]
                ):
                    parser_mode = "structured"

                    file_diffs = compare_flat_structures(
                        old_values=old_parse[
                            "flat_values"
                        ],
                        new_values=new_parse[
                            "flat_values"
                        ],
                    )

                else:
                    parser_mode = "line_fallback"

                    file_diffs = create_line_fallback_diffs(
                        source_before=source_before,
                        source_after=source_after,
                    )

                    for parse_result in [
                        old_parse,
                        new_parse,
                    ]:
                        if (
                            not parse_result["success"]
                            and parse_result["error"]
                        ):
                            error_type = (
                                parse_result["error"]
                                .split(":", maxsplit=1)[0]
                            )

                            parser_error_types[
                                error_type
                            ] += 1

                parser_mode_counts[parser_mode] += 1

                if not file_diffs:
                    file_changes_without_semantic_diffs += 1
                    skip_reasons[
                        "no_semantic_difference"
                    ] += 1

                    progress_bar.set_postfix(
                        diffs=generated_diff_records,
                        structured=parser_mode_counts.get(
                            "structured",
                            0,
                        ),
                        fallback=parser_mode_counts.get(
                            "line_fallback",
                            0,
                        ),
                        errors=processing_errors,
                    )

                    continue

                file_changes_with_diffs += 1

                total_diffs_before_cap = len(file_diffs)

                maximum_diffs = STRUCTURAL_DIFF_SETTINGS[
                    "max_diffs_per_file_change"
                ]

                diff_was_capped = (
                    total_diffs_before_cap
                    > maximum_diffs
                )

                if diff_was_capped:
                    capped_file_changes += 1

                    file_diffs = sorted(
                        file_diffs,
                        key=lambda change: stable_sha256(
                            change["field_path"],
                            change["operation"],
                            canonical_value(
                                change["old_value"]
                            ),
                            canonical_value(
                                change["new_value"]
                            ),
                        ),
                    )[:maximum_diffs]

                emitted_diff_count = len(file_diffs)

                for diff_index, diff in enumerate(
                    file_diffs
                ):
                    old_output_value = truncate_value(
                        diff["old_value"]
                    )

                    new_output_value = truncate_value(
                        diff["new_value"]
                    )

                    diff_id = stable_sha256(
                        source_record["record_id"],
                        diff_index,
                        diff["operation"],
                        diff["field_path"],
                        old_output_value,
                        new_output_value,
                    )

                    change_signature = stable_sha256(
                        configuration_type,
                        diff["operation"],
                        diff["field_path"],
                        old_output_value,
                        new_output_value,
                    )

                    output_record = {
                        "diff_id": diff_id,
                        "source_record_id":
                            source_record["record_id"],
                        "repository": repository_name,
                        "repository_url":
                            source_record["repository_url"],
                        "dataset_split":
                            repository_split,
                        "commit_hash":
                            source_record["commit_hash"],
                        "parent_hash":
                            source_record["parent_hash"],
                        "commit_author_date":
                            source_record[
                                "commit_author_date"
                            ],
                        "commit_message":
                            source_record[
                                "commit_message"
                            ],
                        "author_name":
                            source_record["author_name"],
                        "file_path":
                            effective_path,
                        "old_path":
                            source_record["old_path"],
                        "new_path":
                            source_record["new_path"],
                        "file_change_type":
                            source_record["change_type"],
                        "configuration_type":
                            configuration_type,
                        "parser_name":
                            parser_name,
                        "parser_mode":
                            parser_mode,
                        "old_parse_success":
                            old_parse["success"],
                        "new_parse_success":
                            new_parse["success"],
                        "old_parse_error":
                            (
                                old_parse["error"][:500]
                                if old_parse["error"]
                                else None
                            ),
                        "new_parse_error":
                            (
                                new_parse["error"][:500]
                                if new_parse["error"]
                                else None
                            ),
                        "old_field_count":
                            old_parse["field_count"],
                        "new_field_count":
                            new_parse["field_count"],
                        "operation":
                            diff["operation"],
                        "field_path":
                            diff["field_path"],
                        "old_value":
                            old_output_value,
                        "new_value":
                            new_output_value,
                        "old_value_type":
                            diff["old_value_type"],
                        "new_value_type":
                            diff["new_value_type"],
                        "old_value_length":
                            (
                                len(old_output_value)
                                if old_output_value
                                else 0
                            ),
                        "new_value_length":
                            (
                                len(new_output_value)
                                if new_output_value
                                else 0
                            ),
                        "diff_index_in_file_change":
                            diff_index,
                        "file_diff_count_before_cap":
                            total_diffs_before_cap,
                        "file_diff_count_emitted":
                            emitted_diff_count,
                        "diff_was_capped":
                            diff_was_capped,
                        "source_before_sha256":
                            source_record[
                                "source_before_sha256"
                            ],
                        "source_after_sha256":
                            source_record[
                                "source_after_sha256"
                            ],
                        "change_signature_sha256":
                            change_signature,
                    }

                    output_file.write(
                        json.dumps(
                            output_record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                    generated_diff_records += 1

                    operation_counts[
                        diff["operation"]
                    ] += 1

                    configuration_type_counts[
                        configuration_type
                    ] += 1

            except Exception as error:
                processing_errors += 1

                error_name = type(error).__name__

                skip_reasons[
                    f"processing_error:{error_name}"
                ] += 1

            # Update the visible progress statistics periodically.
            if (
                file_changes_processed % 25 == 0
                or file_changes_processed == 1
            ):
                progress_bar.set_postfix(
                    diffs=generated_diff_records,
                    structured=parser_mode_counts.get(
                        "structured",
                        0,
                    ),
                    fallback=parser_mode_counts.get(
                        "line_fallback",
                        0,
                    ),
                    errors=processing_errors,
                )

        progress_bar.close()

    os.replace(
        temporary_output_path,
        output_path,
    )

    summary = {
        "repository":
            repository_name,
        "dataset_split":
            repository_split,
        "input_path":
            str(input_path),
        "output_path":
            str(output_path),
        "file_changes_processed":
            file_changes_processed,
        "file_changes_with_diffs":
            file_changes_with_diffs,
        "file_changes_without_semantic_diffs":
            file_changes_without_semantic_diffs,
        "generated_diff_records":
            generated_diff_records,
        "capped_file_changes":
            capped_file_changes,
        "processing_errors":
            processing_errors,
        "parser_counts":
            dict(parser_counts),
        "parser_mode_counts":
            dict(parser_mode_counts),
        "operation_counts":
            dict(operation_counts),
        "configuration_type_counts":
            dict(configuration_type_counts),
        "skip_reasons":
            dict(skip_reasons),
        "parser_error_types":
            dict(parser_error_types),
        "generated_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    with summary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
        )

    print("\nRepository completed successfully")
    print(
        "Files processed :",
        f"{file_changes_processed:,}",
    )
    print(
        "Files with diffs:",
        f"{file_changes_with_diffs:,}",
    )
    print(
        "Diff records    :",
        f"{generated_diff_records:,}",
    )
    print(
        "Structured files:",
        f"{parser_mode_counts.get('structured', 0):,}",
    )
    print(
        "Fallback files  :",
        f"{parser_mode_counts.get('line_fallback', 0):,}",
    )
    print(
        "Errors          :",
        f"{processing_errors:,}",
    )
    print("-" * 72)

    return summary


print("Cell 19 loaded successfully.")
print(
    "Function available:",
    callable(generate_structural_diffs_for_repository),
)
print("Run Cell 20 to begin processing.")

Loading structural-diff extraction function...
Cell 19 loaded successfully.
Function available: True
Run Cell 20 to begin processing.


In [24]:
structural_diff_summaries = []

for repository_config in REPOSITORIES:
    print("\n" + "=" * 72)
    print(
        f"Generating structural diffs: "
        f"{repository_config['name']} "
        f"({repository_config['split']})"
    )
    print("=" * 72)

    summary = generate_structural_diffs_for_repository(
        repository_config=repository_config,
        force_regenerate=STRUCTURAL_DIFF_SETTINGS[
            "force_regenerate"
        ],
    )

    structural_diff_summaries.append(summary)

print("\nStructural diff generation completed.")


Generating structural diffs: microservices_demo (train)

Repository : microservices_demo
Split      : train
Input      : microservices_demo_configuration_changes.jsonl.gz
Output     : microservices_demo_structural_diffs.jsonl.gz
Status     : Existing output found. Skipping microservices_demo.

Generating structural diffs: kube_prometheus (train)

Repository : kube_prometheus
Split      : train
Input      : kube_prometheus_configuration_changes.jsonl.gz
Output     : kube_prometheus_structural_diffs.jsonl.gz
Expected file changes: 7,253
Status     : Processing started


kube_prometheus:   0%|          | 0/7253 [00:00<?, ?file/s]


Repository completed successfully
Files processed : 7,253
Files with diffs: 7,166
Diff records    : 130,430
Structured files: 7,041
Fallback files  : 212
Errors          : 0
------------------------------------------------------------------------

Generating structural diffs: terraform_aws_vpc (train)

Repository : terraform_aws_vpc
Split      : train
Input      : terraform_aws_vpc_configuration_changes.jsonl.gz
Output     : terraform_aws_vpc_structural_diffs.jsonl.gz
Expected file changes: 979
Status     : Processing started


terraform_aws_vpc:   0%|          | 0/979 [00:00<?, ?file/s]


Repository completed successfully
Files processed : 979
Files with diffs: 947
Diff records    : 14,505
Structured files: 979
Fallback files  : 0
Errors          : 0
------------------------------------------------------------------------

Generating structural diffs: kubernetes_examples (validation)

Repository : kubernetes_examples
Split      : validation
Input      : kubernetes_examples_configuration_changes.jsonl.gz
Output     : kubernetes_examples_structural_diffs.jsonl.gz
Expected file changes: 1,946
Status     : Processing started


...


Repository completed successfully
Files processed : 1,946
Files with diffs: 1,784
Diff records    : 12,854
Structured files: 1,866
Fallback files  : 80
Errors          : 0
------------------------------------------------------------------------

Generating structural diffs: ansible_examples (test)

Repository : ansible_examples
Split      : test
Input      : ansible_examples_configuration_changes.jsonl.gz
Output     : ansible_examples_structural_diffs.jsonl.gz
Expected file changes: 932
Status     : Processing started


ansible_examples:   0%|          | 0/932 [00:00<?, ?file/s]


Repository completed successfully
Files processed : 932
Files with diffs: 838
Diff records    : 8,940
Structured files: 885
Fallback files  : 47
Errors          : 0
------------------------------------------------------------------------

Generating structural diffs: terraform_eks_blueprints (test)

Repository : terraform_eks_blueprints
Split      : test
Input      : terraform_eks_blueprints_configuration_changes.jsonl.gz
Output     : terraform_eks_blueprints_structural_diffs.jsonl.gz
Expected file changes: 7,700
Status     : Processing started


terraform_eks_blueprints:   0%|          | 0/7700 [00:00<?, ?file/s]


Repository completed successfully
Files processed : 7,700
Files with diffs: 7,070
Diff records    : 109,076
Structured files: 7,548
Fallback files  : 152
Errors          : 0
------------------------------------------------------------------------

Structural diff generation completed.


In [25]:
summary_records = []

for summary in structural_diff_summaries:
    structured_count = summary[
        "parser_mode_counts"
    ].get("structured", 0)

    fallback_count = summary[
        "parser_mode_counts"
    ].get("line_fallback", 0)

    total_parser_attempts = (
        structured_count + fallback_count
    )

    fallback_rate = (
        fallback_count / total_parser_attempts
        if total_parser_attempts > 0
        else 0
    )

    summary_records.append(
        {
            "repository": summary["repository"],
            "split": summary["dataset_split"],
            "file_changes":
                summary["file_changes_processed"],
            "files_with_diffs":
                summary["file_changes_with_diffs"],
            "semantic_no_op_files":
                summary[
                    "file_changes_without_semantic_diffs"
                ],
            "structural_diff_records":
                summary["generated_diff_records"],
            "capped_file_changes":
                summary["capped_file_changes"],
            "structured_parser_files":
                structured_count,
            "fallback_files":
                fallback_count,
            "fallback_rate":
                fallback_rate,
        }
    )

structural_summary_df = pd.DataFrame(
    summary_records
)

display(structural_summary_df)

print(
    "\nTotal structural diff records:",
    f"{structural_summary_df['structural_diff_records'].sum():,}",
)

,repository,split,file_changes,files_with_diffs,semantic_no_op_files,structural_diff_records,capped_file_changes,structured_parser_files,fallback_files,fallback_rate
0,microservices_demo,train,3477,3379,98,83933,25,3137,340,0.097785
1,kube_prometheus,train,7253,7166,87,130430,50,7041,212,0.029229
2,terraform_aws_vpc,train,979,947,32,14505,1,979,0,0.000000
3,kubernetes_examples,validation,1946,1784,162,12854,0,1866,80,0.041110
4,ansible_examples,test,932,838,94,8940,0,885,47,0.050429
5,terraform_eks_blueprints,test,7700,7070,630,109076,15,7548,152,0.019740



Total structural diff records: 359,738


In [26]:
structural_diff_records = []

for repository in REPOSITORIES:
    repository_name = repository["name"]

    structural_diff_path = (
        INTERIM_DATA_DIR
        / f"{repository_name}_structural_diffs.jsonl.gz"
    )

    if not structural_diff_path.exists():
        print(
            "Missing structural diff output:",
            structural_diff_path,
        )
        continue

    with gzip.open(
        structural_diff_path,
        mode="rt",
        encoding="utf-8",
    ) as input_file:

        for line in input_file:
            structural_diff_records.append(
                json.loads(line)
            )

structural_diff_manifest = pd.DataFrame(
    structural_diff_records
)

print(
    "Structural diff manifest shape:",
    structural_diff_manifest.shape,
)

display(structural_diff_manifest.head())

Structural diff manifest shape: (359738, 38)


,diff_id,source_record_id,repository,repository_url,dataset_split,commit_hash,parent_hash,commit_author_date,commit_message,author_name,...,new_value_type,old_value_length,new_value_length,diff_index_in_file_change,file_diff_count_before_cap,file_diff_count_emitted,diff_was_capped,source_before_sha256,source_after_sha256,change_signature_sha256
0,c820a8520734fc4ecbbbcd59610480fca6eb145bfbf626...,f3e711748b8342bed50d50afa170591f3dd0cdd2df05b0...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,github-actions[bot],...,string,7,7,0,2,2,False,0cdfd4aeb97450ca85844360b155948a43e2266793ea25...,b8b3460ce2131db8f4e306ed7d2fd337036390ad1f733f...,d0d237a520586df152592117e701baef3b7bf1dfc96aeb...
1,2778bac3137cbec71eb3ced76ea3d38addea260fd6a570...,f3e711748b8342bed50d50afa170591f3dd0cdd2df05b0...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,github-actions[bot],...,string,6,6,1,2,2,False,0cdfd4aeb97450ca85844360b155948a43e2266793ea25...,b8b3460ce2131db8f4e306ed7d2fd337036390ad1f733f...,2f64170dd0a33c9d4452fec88465e4707574a6a774fdb3...
2,dd772049c2639bec5386ebd1d70965302edda15fd2585f...,4ada560261335ebf7d43ecb68fcd56a2f0f3d380286a6b...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,github-actions[bot],...,string,60,64,0,1,1,False,6fe21455e7f264bf9152fb21ee8454e5d16165c33f924b...,4cc974ac64ec7fb513a6081e267d91448bd76a236e0b4c...,e589654dd270e2d145774d10ef3e177fabd0f658e2a5da...
3,37a016215756d4ed90964a7487ef10b3ed80d5a4d3e8f2...,3cb9882df2d988155d0ea9c634473aa2bb22ba373bad1b...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,github-actions[bot],...,string,78,82,0,1,1,False,53b76b55282942e74e27ff40e168209921e2dcddbbf7b8...,01f805ec9d094304a6a22d6f3f0176a222f242d1592c65...,fdae1e90021ccc89f041eac1e368e71b74d0421f3d1aff...
4,de79a27b115b18a990fedc06348d12fe1e6318c8fe5f7c...,c213b0bf8781befc5354bd284fa14c4cee54b360bb75a5...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,github-actions[bot],...,string,80,84,0,1,1,False,9e628a60c1319fe0dffc7ec94faa3f61ff694cf13d28ab...,e4c19590f89bc737b280739f04d531f42d3f0e9a82e36e...,d866f9226636111d7d8ff83d05e5d2e1ca0681889c45d1...


In [27]:
manifest_csv_path = (
    INTERIM_DATA_DIR
    / "structural_diff_manifest.csv.gz"
)

summary_csv_path = (
    INTERIM_DATA_DIR
    / "structural_diff_repository_summary.csv"
)

summary_json_path = (
    INTERIM_DATA_DIR
    / "structural_diff_repository_summary.json"
)

structural_diff_manifest.to_csv(
    manifest_csv_path,
    index=False,
    compression="gzip",
)

structural_summary_df.to_csv(
    summary_csv_path,
    index=False,
)

with summary_json_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        structural_diff_summaries,
        file,
        indent=2,
    )

print("Saved structural diff manifest:")
print(manifest_csv_path)

print("\nSaved summary:")
print(summary_csv_path)
print(summary_json_path)

Saved structural diff manifest:
C:\Users\Lenovo\Desktop\DriftGuard\data\interim\structural_diff_manifest.csv.gz

Saved summary:
C:\Users\Lenovo\Desktop\DriftGuard\data\interim\structural_diff_repository_summary.csv
C:\Users\Lenovo\Desktop\DriftGuard\data\interim\structural_diff_repository_summary.json


In [28]:
split_summary = (
    structural_diff_manifest
    .groupby("dataset_split")
    .agg(
        diff_records=("diff_id", "count"),
        repositories=("repository", "nunique"),
        commits=("commit_hash", "nunique"),
        files=("file_path", "nunique"),
        source_file_changes=(
            "source_record_id",
            "nunique",
        ),
    )
)

display(split_summary)

print("\nOperation distribution:")

operation_distribution = (
    structural_diff_manifest
    .groupby(
        ["dataset_split", "operation"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "count"})
)

display(operation_distribution)

,diff_records,repositories,commits,files,source_file_changes
dataset_split,,,,,
test,118016,2,1306,2050,7908
train,228868,3,2750,744,11492
validation,12854,1,592,648,1784



Operation distribution:


,dataset_split,operation,count
0,test,added,60030
1,test,deleted,49998
2,test,modified,7988
3,train,added,98642
4,train,deleted,53165
5,train,modified,77061
6,validation,added,8022
7,validation,deleted,3798
8,validation,modified,1034


In [29]:
parser_distribution = (
    structural_diff_manifest
    .groupby(
        [
            "dataset_split",
            "parser_name",
            "parser_mode",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "diff_records"})
    .sort_values(
        [
            "dataset_split",
            "diff_records",
        ],
        ascending=[True, False],
    )
)

print("Parser distribution:")
display(parser_distribution)

technology_distribution = (
    structural_diff_manifest
    .groupby(
        [
            "dataset_split",
            "configuration_type",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "diff_records"})
    .sort_values(
        [
            "dataset_split",
            "diff_records",
        ],
        ascending=[True, False],
    )
)

print("\nConfiguration technology distribution:")
display(technology_distribution)

Parser distribution:


,dataset_split,parser_name,parser_mode,diff_records
3,test,hcl,structured,72227
8,test,yaml,structured,25274
4,test,json,structured,10717
7,test,yaml,line_fallback,8991
2,test,hcl,line_fallback,486
1,test,generic_key_value,structured,139
0,test,dockerfile,structured,116
5,test,nginx,structured,60
6,test,toml,structured,6
14,train,json,structured,93515



Configuration technology distribution:


,dataset_split,configuration_type,diff_records
7,test,terraform,72713
9,test,yaml,15233
5,test,kubernetes,10042
4,test,json,9291
0,test,ansible,6165
6,test,nginx,2516
3,test,helm,1795
1,test,configuration,139
2,test,docker,116
8,test,toml,6


In [30]:
sample_columns = [
    "repository",
    "dataset_split",
    "commit_hash",
    "configuration_type",
    "parser_name",
    "parser_mode",
    "operation",
    "file_path",
    "field_path",
    "old_value",
    "new_value",
]

structured_samples = (
    structural_diff_manifest[
        structural_diff_manifest[
            "parser_mode"
        ] == "structured"
    ]
    .sample(
        n=min(
            20,
            len(
                structural_diff_manifest[
                    structural_diff_manifest[
                        "parser_mode"
                    ] == "structured"
                ]
            ),
        ),
        random_state=42,
    )
)

display(
    structured_samples[sample_columns]
)

,repository,dataset_split,commit_hash,configuration_type,parser_name,parser_mode,operation,file_path,field_path,old_value,new_value
38705,microservices_demo,train,f75fde0f59505f7498439f86b142091510c1420a,json,json,structured,modified,src/currencyservice/package-lock.json,$.packages.<empty>.dependencies.@opentelemetry...,1.15.0,1.15.1
327080,terraform_eks_blueprints,test,2a08e0753dfcfdd36a68840b66357628b893fee5,terraform,hcl,structured,added,modules/kubernetes-addons/helm_addon/main.tf,$.module irsa[0].tags[0],NaN,var.irsa_config.tags
143238,kube_prometheus,train,f36b68458db8b8e4f95bc7d01430dcdc78b5e9cc,yaml,yaml,structured,added,manifests/alertmanager-secret.yaml,$.metadata.labels.app.kubernetes.io/name,NaN,alertmanager
229714,kubernetes_examples,validation,27216de59e007f4a88b65256bbb1cb30c53662b3,json,json,structured,added,staging/scheduler-policy/scheduler-policy-conf...,$.extenders[0].weight,NaN,5
19460,microservices_demo,train,dd0c4e14126f5d5f7ebf192b6c7f0a22afbc9ac8,json,json,structured,modified,src/paymentservice/package-lock.json,$.dependencies.@opentelemetry/sdk-trace-node.r...,https://registry.npmjs.org/@opentelemetry/sdk-...,https://registry.npmjs.org/@opentelemetry/sdk-...
333542,terraform_eks_blueprints,test,479b1a909c07fdc072181056bdcb799d90d8a877,terraform,hcl,structured,added,modules/kubernetes-addons/traefik/main.tf,$.resource helm_release traefik[0].repository[0],NaN,"local.helm_config[""repository""]"
153751,kube_prometheus,train,926337feacef28c43a079ba4045ed385e7243532,yaml,yaml,structured,modified,manifests/prometheus-operator-serviceMonitor.yaml,$.metadata.labels.app.kubernetes.io/version,v0.38.0,v0.38.1
211487,kube_prometheus,train,ec79cbcbda3593db36292cc2ac2538cd970e18ef,json,json,structured,added,assets/grafana/all-nodes.json,$.rows[2].panels[1].mappingTypes[1].value,NaN,2
313809,terraform_eks_blueprints,test,deec7d5caea47c06dea48fa616ad2c56e52c3cce,terraform,hcl,structured,added,examples/eks-cluster-with-new-vpc/versions.tf,$.terraform[0].required_version[0],NaN,""">= 1.0.0"""
23778,microservices_demo,train,672f2b2576a57db1bd52650695ba0d687c8e8884,json,json,structured,modified,src/paymentservice/package-lock.json,$.packages.node_modules/@opentelemetry/otlp-tr...,1.25.0,1.25.1


In [31]:
SENSITIVE_KEYWORDS = [
    "ssl",
    "tls",
    "auth",
    "password",
    "secret",
    "token",
    "credential",
    "port",
    "listen",
    "bind",
    "acl",
    "allow",
    "deny",
    "access",
    "firewall",
    "security",
    "privileged",
    "capabilities",
    "limit",
    "quota",
    "encryption",
]

sensitive_pattern = "|".join(
    re.escape(keyword)
    for keyword in SENSITIVE_KEYWORDS
)

sensitive_candidates = structural_diff_manifest[
    structural_diff_manifest[
        "field_path"
    ].str.contains(
        sensitive_pattern,
        case=False,
        na=False,
        regex=True,
    )
    |
    structural_diff_manifest[
        "new_value"
    ].fillna("").str.contains(
        sensitive_pattern,
        case=False,
        regex=True,
    )
].copy()

print(
    "Potentially security-sensitive diffs:",
    f"{len(sensitive_candidates):,}",
)

display(
    sensitive_candidates[
        sample_columns
    ].head(30)
)

Potentially security-sensitive diffs: 63,852


,repository,dataset_split,commit_hash,configuration_type,parser_name,parser_mode,operation,file_path,field_path,old_value,new_value
53,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,added,release/istio-manifests.yaml,$.documents[0].spec.http[0].route[0].destinati...,NaN,80
54,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,deleted,release/istio-manifests.yaml,$.documents[0].spec.listeners[0].allowedRoutes...,Same,NaN
55,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,deleted,release/istio-manifests.yaml,$.documents[0].spec.listeners[0].name,http,NaN
56,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,deleted,release/istio-manifests.yaml,$.documents[0].spec.listeners[0].port,80,NaN
57,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,deleted,release/istio-manifests.yaml,$.documents[0].spec.listeners[0].protocol,HTTP,NaN
61,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,added,release/istio-manifests.yaml,$.documents[1].spec.listeners[0].allowedRoutes...,NaN,Same
62,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,added,release/istio-manifests.yaml,$.documents[1].spec.listeners[0].name,NaN,http
63,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,added,release/istio-manifests.yaml,$.documents[1].spec.listeners[0].port,NaN,80
64,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,added,release/istio-manifests.yaml,$.documents[1].spec.listeners[0].protocol,NaN,HTTP
67,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,yaml,yaml,structured,deleted,release/istio-manifests.yaml,$.documents[1].spec.rules[0].backendRefs[0].port,80,NaN


In [32]:
repositories_by_split = (
    structural_diff_manifest
    .groupby("dataset_split")[
        "repository"
    ]
    .apply(set)
    .to_dict()
)

train_repositories = repositories_by_split.get(
    "train",
    set(),
)

validation_repositories = repositories_by_split.get(
    "validation",
    set(),
)

test_repositories = repositories_by_split.get(
    "test",
    set(),
)

split_integrity_checks = {
    "Train and validation repositories are disjoint":
        train_repositories.isdisjoint(
            validation_repositories
        ),

    "Train and test repositories are disjoint":
        train_repositories.isdisjoint(
            test_repositories
        ),

    "Validation and test repositories are disjoint":
        validation_repositories.isdisjoint(
            test_repositories
        ),
}

for check_name, passed in split_integrity_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status:<8} | {check_name}")

assert all(split_integrity_checks.values())

print("\nTrain repositories:")
print(sorted(train_repositories))

print("\nValidation repositories:")
print(sorted(validation_repositories))

print("\nUnseen test repositories:")
print(sorted(test_repositories))

PASSED   | Train and validation repositories are disjoint
PASSED   | Train and test repositories are disjoint
PASSED   | Validation and test repositories are disjoint

Train repositories:
['kube_prometheus', 'microservices_demo', 'terraform_aws_vpc']

Validation repositories:
['kubernetes_examples']

Unseen test repositories:
['ansible_examples', 'terraform_eks_blueprints']


In [33]:
integrity_checks = {
    "Structural manifest is not empty":
        len(structural_diff_manifest) > 0,

    "Every diff ID is unique":
        structural_diff_manifest[
            "diff_id"
        ].is_unique,

    "Every source record ID is present":
        structural_diff_manifest[
            "source_record_id"
        ].notna().all(),

    "Every diff has a repository":
        structural_diff_manifest[
            "repository"
        ].notna().all(),

    "Every diff has a commit hash":
        structural_diff_manifest[
            "commit_hash"
        ].notna().all(),

    "Every diff has a file path":
        structural_diff_manifest[
            "file_path"
        ].notna().all(),

    "Every diff has a field path":
        structural_diff_manifest[
            "field_path"
        ].notna().all(),

    "Every operation is valid":
        structural_diff_manifest[
            "operation"
        ].isin(
            [
                "added",
                "deleted",
                "modified",
            ]
        ).all(),

    "Every split is valid":
        structural_diff_manifest[
            "dataset_split"
        ].isin(
            [
                "train",
                "validation",
                "test",
            ]
        ).all(),

    "Training records exist":
        (
            structural_diff_manifest[
                "dataset_split"
            ] == "train"
        ).any(),

    "Validation records exist":
        (
            structural_diff_manifest[
                "dataset_split"
            ] == "validation"
        ).any(),

    "Unseen test records exist":
        (
            structural_diff_manifest[
                "dataset_split"
            ] == "test"
        ).any(),

    "Change signatures are available":
        structural_diff_manifest[
            "change_signature_sha256"
        ].notna().all(),
}

print("Data-integrity checks:\n")

for check_name, passed in integrity_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status:<8} | {check_name}")

all_checks_passed = (
    all(integrity_checks.values())
    and all(split_integrity_checks.values())
)

print("\n" + "=" * 72)

if all_checks_passed:
    print("ALL STRUCTURAL DIFF CHECKS PASSED")
else:
    print("ONE OR MORE STRUCTURAL DIFF CHECKS FAILED")

print("=" * 72)

Data-integrity checks:

PASSED   | Structural manifest is not empty
PASSED   | Every diff ID is unique
PASSED   | Every source record ID is present
PASSED   | Every diff has a repository
PASSED   | Every diff has a commit hash
PASSED   | Every diff has a file path
PASSED   | Every diff has a field path
PASSED   | Every operation is valid
PASSED   | Every split is valid
PASSED   | Training records exist
PASSED   | Validation records exist
PASSED   | Unseen test records exist
PASSED   | Change signatures are available

ALL STRUCTURAL DIFF CHECKS PASSED


In [34]:
parser_mode_file_totals = Counter()

for summary in structural_diff_summaries:
    for mode, count in summary[
        "parser_mode_counts"
    ].items():
        parser_mode_file_totals[mode] += count

structured_file_count = parser_mode_file_totals.get(
    "structured",
    0,
)

fallback_file_count = parser_mode_file_totals.get(
    "line_fallback",
    0,
)

total_parsed_file_changes = (
    structured_file_count
    + fallback_file_count
)

overall_fallback_rate = (
    fallback_file_count
    / total_parsed_file_changes
    if total_parsed_file_changes > 0
    else 0
)

print("Structured parser file changes:", structured_file_count)
print("Fallback file changes         :", fallback_file_count)
print("Total parser attempts         :", total_parsed_file_changes)
print(
    "Overall fallback rate         :",
    f"{overall_fallback_rate:.2%}",
)

if overall_fallback_rate <= 0.25:
    print("\nParser coverage status: STRONG")
elif overall_fallback_rate <= 0.50:
    print("\nParser coverage status: ACCEPTABLE")
else:
    print(
        "\nParser coverage status: HIGH FALLBACK RATE — "
        "review parser errors before model training."
    )

Structured parser file changes: 21456
Fallback file changes         : 831
Total parser attempts         : 22287
Overall fallback rate         : 3.73%

Parser coverage status: STRONG


In [35]:
signature_split_counts = (
    structural_diff_manifest
    .groupby(
        "change_signature_sha256"
    )["dataset_split"]
    .nunique()
)

cross_split_signatures = signature_split_counts[
    signature_split_counts > 1
]

cross_split_signature_count = len(
    cross_split_signatures
)

unique_signature_count = (
    structural_diff_manifest[
        "change_signature_sha256"
    ].nunique()
)

cross_split_rate = (
    cross_split_signature_count
    / unique_signature_count
    if unique_signature_count > 0
    else 0
)

print(
    "Unique change signatures:",
    f"{unique_signature_count:,}",
)

print(
    "Signatures appearing across multiple splits:",
    f"{cross_split_signature_count:,}",
)

print(
    "Cross-split signature rate:",
    f"{cross_split_rate:.2%}",
)

print(
    "\nThese overlaps are not removed in this notebook. "
    "They will be grouped or deduplicated before model training."
)

Unique change signatures: 223,490
Signatures appearing across multiple splits: 1,225
Cross-split signature rate: 0.55%

These overlaps are not removed in this notebook. They will be grouped or deduplicated before model training.


In [36]:
total_diff_records = len(
    structural_diff_manifest
)

total_unique_commits = (
    structural_diff_manifest[
        "commit_hash"
    ].nunique()
)

total_unique_file_changes = (
    structural_diff_manifest[
        "source_record_id"
    ].nunique()
)

total_unique_fields = (
    structural_diff_manifest[
        "field_path"
    ].nunique()
)

structured_record_count = (
    structural_diff_manifest[
        "parser_mode"
    ]
    .eq("structured")
    .sum()
)

fallback_record_count = (
    structural_diff_manifest[
        "parser_mode"
    ]
    .eq("line_fallback")
    .sum()
)

print("=" * 72)
print("NOTEBOOK 03 COMPLETED")
print("=" * 72)

print(
    "Field-level diff records       :",
    f"{total_diff_records:,}",
)

print(
    "Unique configuration commits   :",
    f"{total_unique_commits:,}",
)

print(
    "Source file changes represented:",
    f"{total_unique_file_changes:,}",
)

print(
    "Unique field paths             :",
    f"{total_unique_fields:,}",
)

print(
    "Structured diff records        :",
    f"{structured_record_count:,}",
)

print(
    "Fallback line-diff records     :",
    f"{fallback_record_count:,}",
)

print(
    "Security-sensitive candidates  :",
    f"{len(sensitive_candidates):,}",
)

print("\nDataset summary by split:")
display(split_summary)

print("\nImportant:")
print("- The test repositories remain fully unseen.")
print("- No risk labels have been assigned yet.")
print("- Parser failures were retained using line-level fallback.")
print("- Cross-split duplicate signatures will be removed before training.")

print("\nNext notebook:")
print("04_security_label_generation.ipynb")

NOTEBOOK 03 COMPLETED
Field-level diff records       : 359,738
Unique configuration commits   : 4,648
Source file changes represented: 21,184
Unique field paths             : 76,590
Structured diff records        : 322,206
Fallback line-diff records     : 37,532
Security-sensitive candidates  : 63,852

Dataset summary by split:


,diff_records,repositories,commits,files,source_file_changes
dataset_split,,,,,
test,118016,2,1306,2050,7908
train,228868,3,2750,744,11492
validation,12854,1,592,648,1784



Important:
- The test repositories remain fully unseen.
- No risk labels have been assigned yet.
- Parser failures were retained using line-level fallback.
- Cross-split duplicate signatures will be removed before training.

Next notebook:
04_security_label_generation.ipynb
